# WatermarkingForLLM on Google Colab

Clones this **public** repo, builds **CPRF** (Go) and **PRC** (Rust/maturin), installs Python deps, and runs **`app.py`** (single-run protocol demo) or **`benchmark_policy_detection.py`** (many trials, aggregated metrics).

**Optional:** Put a **`.env`** in the repo root on the VM with **`HF_TOKEN=`** so section 5 can log in non-interactively; otherwise use **`notebook_login()`** there.

**Before you start:** **Runtime → Change runtime type → GPU**. Use **Python 3.11+** if offered.

The first code cell enables **IPython autoreload** so that after **`git pull`**, re-running later cells picks up changes to **`watermarking.py`** and other modules without restarting the kernel.


In [1]:
import sys

assert sys.version_info >= (3, 11), "Use Python 3.11+ (Runtime → Change runtime type)"

import torch

print("Python:", sys.version.split()[0])
print("CUDA:", torch.cuda.is_available(), end="")
if torch.cuda.is_available():
    print(" —", torch.cuda.get_device_name(0))
else:
    print()

# After ``git pull``, edited repo modules reload when you re-run downstream cells (no kernel restart).
import types

if sys.version_info >= (3, 12):
    # IPython's bundled ``autoreload`` still does ``from imp import reload``; ``imp`` was removed in 3.12.
    try:
        import imp as _imp_check  # noqa: F401
    except ModuleNotFoundError:
        import importlib

        _imp_stub = types.ModuleType("imp")
        _imp_stub.reload = importlib.reload
        sys.modules.setdefault("imp", _imp_stub)

try:
    _ipy = get_ipython()  # type: ignore[name-defined]
    _ipy.run_line_magic("load_ext", "autoreload")
    _ipy.run_line_magic("autoreload", "2")
    print("IPython: autoreload 2 enabled.")
except NameError:
    print("(autoreload skipped: not in IPython/Jupyter)")
except ModuleNotFoundError as _e:
    if getattr(_e, "name", None) == "imp" or "No module named 'imp'" in str(_e):
        print(
            "(autoreload failed: install a newer IPython, e.g. %pip install -U ipython)",
            repr(_e),
        )
    else:
        raise


Python: 3.12.13
CUDA: True — NVIDIA L4


## 1. Clone the repo

Edit **`REPO_OWNER`** / **`REPO_NAME`** in the next cell if you use a fork. The tree is cloned to **`/content/<REPO_NAME>`**.

If you add **`.env`** on the VM (e.g. for Hugging Face), section 1 loads it automatically after cloning.


In [2]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# --- edit if you use a fork ---
REPO_OWNER = "maxraffel"
REPO_NAME = "attribute-based-watermarking"

PROJECT_ROOT = Path("/content") / REPO_NAME
CLONE_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not (PROJECT_ROOT / "watermarking.py").is_file():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(
        ["git", "clone", "--depth", "1", CLONE_URL, str(PROJECT_ROOT)],
        check=True,
    )
else:
    print("Repo already present:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
_env = PROJECT_ROOT / ".env"
if _env.is_file():
    try:
        from dotenv import load_dotenv
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "python-dotenv"],
            check=True,
        )
        from dotenv import load_dotenv

    load_dotenv(_env)
    print("Loaded", _env)

print("cwd:", PROJECT_ROOT.resolve())

# --- Hugging Face hub id for the watermark causal LM (edit here; used in sections 7 and 8) ---
WATERMARK_LLM_ID = "meta-llama/Llama-3.2-1B-Instruct"


cwd: /content/attribute-based-watermarking


## 2. Build CPRF (`cprf.so`)

Linux `.so` only. The next cell aligns `go.mod` to **go 1.23** when needed.
**Go installs from https://go.dev/dl** (official tarball → `/usr/local/go`) when `go` is missing — avoids `apt-get update` hangs on Colab. `apt-get` runs only if the download fails (with timeouts). If nothing works: **Runtime → Restart session** and retry.


In [3]:
import os
import platform
import shutil
import subprocess
from pathlib import Path

cprf_dir = PROJECT_ROOT / "cprf"
so_path = cprf_dir / "cprf.so"

# Colab's apt golang is often older than the repo's `go` directive; relax to 1.23 for the build.
_go_mod = cprf_dir / "go.mod"
if _go_mod.is_file():
    _lines = _go_mod.read_text(encoding="utf-8").splitlines(keepends=True)
    _out, _changed = [], False
    for _line in _lines:
        _s = _line.strip()
        if _s.startswith("go ") and not _s.startswith("go 1.23"):
            _out.append("go 1.23\n")
            _changed = True
        else:
            _out.append(_line)
    if _changed:
        _go_mod.write_text("".join(_out), encoding="utf-8")


def _prepend_path(bin_dir: str) -> None:
    os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")


def _ensure_go(timeout_s: int = 420) -> None:
    """Prefer official Go tarball (avoids apt hangs on Colab); bounded apt as fallback."""
    if shutil.which("go") is not None:
        return
    env = dict(os.environ)
    env.setdefault("DEBIAN_FRONTEND", "noninteractive")
    sub = dict(check=True, stdin=subprocess.DEVNULL, env=env, timeout=timeout_s)

    prefix = Path("/usr/local")
    goroot = prefix / "go"
    bindir = str(goroot / "bin")
    go_exe = goroot / "bin" / "go"
    if go_exe.is_file():
        _prepend_path(bindir)
        subprocess.run(
            [str(go_exe), "version"],
            check=True,
            stdin=subprocess.DEVNULL,
            timeout=120,
        )
        return

    GO_VER = "1.23.4"
    uarch_map = {"x86_64": "amd64", "AMD64": "amd64", "aarch64": "arm64", "arm64": "arm64"}
    arch = uarch_map.get(platform.machine(), "amd64")
    tgz_name = f"go{GO_VER}.linux-{arch}.tar.gz"
    tgz_path = Path("/tmp") / tgz_name
    dl_url = f"https://go.dev/dl/{tgz_name}"

    print("go missing: fetching", dl_url, "(tarball — avoids slow apt)\n", flush=True)
    curl_kw = dict(check=False, stdin=subprocess.DEVNULL, env=env, timeout=timeout_s)
    dl_ok = False
    if shutil.which("curl"):
        r = subprocess.run(
            [
                "curl", "-fL", "--retry", "3", "--connect-timeout", "30",
                "--max-time", str(timeout_s), "-o", str(tgz_path), dl_url,
            ],
            **curl_kw,
        )
        dl_ok = r.returncode == 0 and tgz_path.is_file() and tgz_path.stat().st_size > 1024 * 1024
    elif shutil.which("wget"):
        r = subprocess.run(
            ["wget", "-nv", "--timeout=120", "--tries=3", "-O", str(tgz_path), dl_url],
            **curl_kw,
        )
        dl_ok = r.returncode == 0 and tgz_path.is_file() and tgz_path.stat().st_size > 1024 * 1024

    if dl_ok:
        if goroot.is_dir():
            shutil.rmtree(goroot)
        subprocess.run(["tar", "-C", str(prefix), "-xzf", str(tgz_path)], **sub)
        _prepend_path(bindir)
        if not go_exe.is_file():
            raise FileNotFoundError(f"Tarball unpacked but missing {go_exe}")
        subprocess.run(
            [str(go_exe), "version"],
            check=True,
            stdin=subprocess.DEVNULL,
            timeout=120,
        )
        return

    print("Tarball failed; trying apt-get with timeout…\n", flush=True)
    try:
        subprocess.run(
            ["apt-get", "update", "-qq", "-o", "Acquire::Retries=3"],
            check=True,
            stdin=subprocess.DEVNULL,
            env=env,
            timeout=timeout_s,
        )
        subprocess.run(
            [
                "apt-get",
                "install",
                "-y",
                "-qq",
                "-o",
                "Dpkg::Use-Pty=0",
                "-o",
                "Acquire::Retries=3",
                "--no-install-recommends",
                "golang-go",
            ],
            check=True,
            stdin=subprocess.DEVNULL,
            env=env,
            timeout=timeout_s,
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError(
            "apt-get exceeded timeout — restart Runtime, or install golang-go in a shell, then re-run."
        ) from None


_ensure_go()

_go_exe = shutil.which("go")
if _go_exe is None and (Path("/usr/local/go/bin/go")).is_file():
    _go_exe = "/usr/local/go/bin/go"
if _go_exe is None:
    raise FileNotFoundError("go executable not found after _ensure_go(); check tarball unpack.")

if not so_path.is_file():
    subprocess.run(
        [_go_exe, "build", "-o", "cprf.so", "-buildmode=c-shared", "cprf.go"],
        cwd=cprf_dir,
        check=True,
    )

assert so_path.is_file(), "cprf.so missing"
print("CPRF:", so_path)


go missing: fetching https://go.dev/dl/go1.23.4.linux-amd64.tar.gz (tarball — avoids slow apt)

CPRF: /content/attribute-based-watermarking/cprf/cprf.so


## 3. Build PRC (Rust + maturin)

Installs Rust in the VM home if `rustc` is missing, then **`maturin build`** (release wheel) and **`pip install`** that wheel. This avoids `maturin develop` issues on hosted Colab; build stdout/stderr are printed when something fails.

In [4]:
import os
import subprocess
import sys
from pathlib import Path

cargo_bin = Path.home() / ".cargo" / "bin"
if not (cargo_bin / "rustc").exists():
    subprocess.run(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y",
        shell=True,
        check=True,
    )
os.environ["PATH"] = str(cargo_bin) + os.pathsep + os.environ.get("PATH", "")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)

try:
    print("Building Rust package with maturin build...")
    completed_process = subprocess.run(
        ["maturin", "build", "--release", "-m", "prc/Cargo.toml"],
        cwd=PROJECT_ROOT,
        check=False,
        capture_output=True,
        text=True,
    )

    if completed_process.stdout:
        print("Maturin build stdout:\n", completed_process.stdout)
    if completed_process.stderr:
        print("Maturin build stderr:\n", completed_process.stderr)

    if completed_process.returncode != 0:
        raise subprocess.CalledProcessError(
            completed_process.returncode,
            completed_process.args,
            output=completed_process.stdout,
            stderr=completed_process.stderr,
        )

    wheel_dir = PROJECT_ROOT / "prc" / "target" / "wheels"
    wheel_files = sorted(
        wheel_dir.glob("*.whl"), key=lambda p: p.stat().st_mtime, reverse=True
    )
    wheel_file = wheel_files[0] if wheel_files else None

    if wheel_file:
        print(f"Installing {wheel_file.name} with pip...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", str(wheel_file)],
            check=True,
        )
    else:
        raise FileNotFoundError("No wheel file found after maturin build.")

except subprocess.CalledProcessError as e:
    print("Maturin failed with error:", e)
    print("Standard Output:", e.stdout)
    print("Standard Error:", e.stderr)
    raise
except FileNotFoundError as e:
    print(e)
    raise

print("PRC: maturin build and pip install ok")

Building Rust package with maturin build...
Maturin build stderr:
     Updating crates.io index
  Downloaded byteorder v1.5.0
  Downloaded generic-array v0.14.7
  Downloaded base64 v0.22.1
  Downloaded serde_derive v1.0.218
  Downloaded zerocopy v0.7.35
  Downloaded serde v1.0.218
  Downloaded version_check v0.9.5
  Downloaded unindent v0.2.3
  Downloaded memoffset v0.9.1
  Downloaded pyo3-ffi v0.23.3
  Downloaded zerocopy-derive v0.7.35
  Downloaded proc-macro2 v1.0.92
  Downloaded rand v0.8.5
  Downloaded syn v2.0.90
  Downloaded unicode-ident v1.0.14
  Downloaded once_cell v1.20.2
  Downloaded pyo3-build-config v0.23.3
  Downloaded pem v3.0.5
  Downloaded rand_core v0.6.4
  Downloaded quote v1.0.37
  Downloaded pyo3-macros v0.23.3
  Downloaded ppv-lite86 v0.2.20
  Downloaded inout v0.1.3
  Downloaded indoc v2.0.5
  Downloaded heck v0.5.0
  Downloaded portable-atomic v1.10.0
  Downloaded lazy_static v1.5.0
  Downloaded rand_chacha v0.3.1
  Downloaded pyo3-macros-backend v0.23.3
  Dow

## 4. Python dependencies (Colab `%pip`)

Uses Colab’s recommended install path. Colab usually ships **PyTorch with CUDA**; extra packages match `pyproject.toml` (without the local-only CUDA index).

The install cell also force-reinstalls **`sympy`** first to avoid a Colab state issue that can surface as:
- `AttributeError: module 'sympy' has no attribute 'core'`
- follow-on `transformers` import failures (`GenerationMixin`, `AutoTokenizer`).

In [23]:
# Colab occasionally ships a broken/stale sympy state that breaks transformers imports
# (e.g. "module 'sympy' has no attribute 'core'"). Force a clean sympy reinstall first.
%pip install -q --upgrade --force-reinstall "sympy>=1.13,<2"
%pip install -q "transformers==5.5.4" "rich>=15" "keybert>=0.9" sentencepiece accelerate python-dotenv

## 5. Hugging Face login (gated Llama)

Accept the **Llama 3.2** license on the model card.

If **`HF_TOKEN`** or **`HUGGING_FACE_HUB_TOKEN`** is set (environment or **`PROJECT_ROOT/.env`** loaded in section 1 or below), you are logged in non-interactively. Otherwise the cell uses **`notebook_login()`**.


In [5]:
import os
from pathlib import Path

from huggingface_hub import login, notebook_login

try:
    _root = PROJECT_ROOT
except NameError:
    _root = Path("/content") / "attribute-based-watermarking"

_env = _root / ".env"
if _env.is_file():
    try:
        from dotenv import load_dotenv

        load_dotenv(_env)
    except ImportError:
        pass

_token = (
    os.environ.get("HF_TOKEN", "").strip()
    or os.environ.get("HUGGING_FACE_HUB_TOKEN", "").strip()
)
if _token:
    login(token=_token, add_to_git_credential=False)
    print("HF: logged in from environment token.")
else:
    notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 6. Pull latest from GitHub

Run after section 1 when you want the newest commit (**public** remote, no token).

Re-run CPRF (section 2) / PRC (section 3) only if those directories changed or builds fail after an update.


In [6]:
import subprocess
from pathlib import Path

REPO_OWNER = "maxraffel"
REPO_NAME = "attribute-based-watermarking"
PROJECT_ROOT = Path("/content") / REPO_NAME

if not (PROJECT_ROOT / ".git").is_dir():
    raise FileNotFoundError("Run section 1 first.")

subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
log = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--oneline"],
    capture_output=True,
    text=True,
    check=True,
)
print(log.stdout.strip())


2fe81ea argmax from partition instead of sample


## 7. Run the demo (`app.py`)

Set **`WATERMARK_LLM_ID`** in **section 1** (clone/setup cell). The next cell imports **`watermarking`** and calls **`watermarking.set_llm_model_id(WATERMARK_LLM_ID)`** before **`runpy.run_path(app.py)`** so the notebook picks the LM without environment variables.

The next cell also sets **`APP_CODE_LENGTH`** and **`APP_WM_BIT_REDUNDANCY`** (integers) and pushes them into **`os.environ`** so **`app.py`** picks them up without editing the file (same keys work from any shell).

To override only for this demo, assign **`WATERMARK_LLM_ID`** again at the top of the next cell before **`import watermarking`**.

The demo follows the current **`app.py`** flow: generate → verify **`derive_x`** on the watermarked text → unconstrained key plus **one constrained key per closed-vocabulary label** → CPRF checks → **`master_detect` / `detect`** vs NLI-active expectations → **negative control** on a decoy transcript. Also loads **DeBERTa-v3-large** NLI for ``derive_x``. First run downloads both; VRAM use can be high on a T4.


In [7]:
import os
import runpy
import sys

# Requires ``PROJECT_ROOT`` from the setup cell (section 1).
try:
    PROJECT_ROOT
except NameError as e:
    raise RuntimeError(
        "PROJECT_ROOT is not defined; run the repo setup cell (section 1) first."
    ) from e

os.chdir(PROJECT_ROOT)

# PRC / channel (read by app.py via os.environ; edit here for Colab without changing app.py).
APP_CODE_LENGTH = 300
APP_WM_BIT_REDUNDANCY = 1
os.environ["APP_CODE_LENGTH"] = str(int(APP_CODE_LENGTH))
os.environ["APP_WM_BIT_REDUNDANCY"] = str(int(APP_WM_BIT_REDUNDANCY))

# Optional: override the hub id from section 1 for this cell only.
WATERMARK_LLM_ID = "meta-llama/Llama-3.2-1B-Instruct"

try:
    _llm_demo = WATERMARK_LLM_ID
except NameError as e:
    raise RuntimeError(
        "WATERMARK_LLM_ID is not defined; set it in section 1 (clone/setup cell) before running this cell."
    ) from e

import watermarking as wm

wm.set_llm_model_id(_llm_demo)

# ``app.py`` ends with ``raise SystemExit(main())``. Jupyter shows that as an exception
# traceback even when the script only returns a nonzero exit code (failed checks).
try:
    runpy.run_path(str(PROJECT_ROOT / "app.py"), run_name="__main__")
except SystemExit as exc:
    code = exc.code
    if code not in (0, None):
        print("app.py exited with code", repr(code), "(0 means all protocol checks passed).")


Loading causal LM 'meta-llama/Llama-3.2-1B-Instruct' on cuda


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

(truncated to 400 chars)
Unrelated decoy text used only as a negative control. Unrelated decoy text used only as a negative control. 
Unrelated decoy text used only as a negative control. Unrelated decoy text used only as a negative control. 
Unrelated decoy text used only as a negative control. Unrelated decoy text used only as a negative control. 
Unrelated decoy text used only as a negative control. Unrelated decoy text u

master_detect → False  recovered: 1100111000110100111000111111000001111110111101000111001100111000... (300 total)

PASS  master_detect(wrong transcript) should be False  (got False, expect False)

╭───────────────────────────────────────────────────╮
│ One or more checks failed — see FAIL lines above. │
╰───────────────────────────────────────────────────╯

app.py exited with code 1 (0 means all protocol checks passed).


## 8. Run the policy benchmark (`benchmark_policy_detection.py`)

The benchmark matches the **same end-to-end protocol as `app.py`**: generate → verify **`derive_x`** → unconstrained + **per-vocabulary-word** constrained keys → CPRF seed checks → **`master_detect` / `detect`** (expected pass when the required label is in the verify-time NLI-active set) → **negative control** (`master_detect` on a decoy transcript should fail). It does **not** print verbose per-trial logs; it shows a **Rich progress bar** and **two summary tables**: per-prompt **TPR/FNR/TNR/FPR** (over `runs × |V|` label decisions), scheme match counts, mismatch diagnostics, and **mean wall time per pipeline stage**.

Edit the **`BENCHMARK_*`** constants in the next cell (including **`BENCHMARK_WM_BIT_REDUNDANCY`**, same semantics as **`app.py`**). The cell calls **`run_benchmark(...)`** directly (same as the CLI), so every flag has a named parameter.

**Prompt list:** `BENCHMARK_PROMPT_CASES` — each string is `"id:prompt"` (only the first `:` splits). If **empty**, **`benchmark_policy_detection.DEFAULT_PROMPT_CASES`** is used (same default prompt shape as `app.py`).

**Causal LM:** set **`BENCHMARK_LLM_MODEL`** to a hub id string, or leave **`None`** to use **`WATERMARK_LLM_ID`** from section 1 when defined.

**Tables in logs:** set **`BENCHMARK_PLAIN_TABLE`** to **`True`** for ASCII tables (e.g. captured logs); **`False`** keeps Rich tables when stdout is a TTY. You can also set the environment variable **`BENCHMARK_PLAIN_TABLE`** before running the cell.

Assume **section 1** defined **`PROJECT_ROOT`**. Run **section 7** first if you want warm HF caches; the benchmark reuses the same **`watermarking`** stack.


In [ ]:
import os
import sys
from pathlib import Path

# --- benchmark parameters (mirror CLI: uv run python benchmark_policy_detection.py ...) ---
BENCHMARK_RUNS = 10
BENCHMARK_CODE_LENGTH = 300
BENCHMARK_WM_BIT_REDUNDANCY = 1
BENCHMARK_MODULUS = 1024
# True: reuse one CPRF master key per prompt id across runs; False: fresh key every trial (default).
BENCHMARK_REUSE_KEY = False

# Hub id for the causal LM, or None to use WATERMARK_LLM_ID from section 1 when set.
BENCHMARK_LLM_MODEL: str | None = None

# True: ASCII tables via BENCHMARK_PLAIN_TABLE=1 (good for captured logs). False: leave env / TTY behavior.
BENCHMARK_FORCE_PLAIN_TABLE = False

# Each "id:prompt" (split on first ':' only). Empty list -> benchmark_policy_detection.DEFAULT_PROMPT_CASES.
BENCHMARK_PROMPT_CASES: list[str] = [
    # "pickleball:Summarize the rules of pickleball for a beginner.",
]

# ---------------------------------------------------------------------------

os.chdir(PROJECT_ROOT)
_root = Path(str(PROJECT_ROOT))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

if BENCHMARK_FORCE_PLAIN_TABLE:
    os.environ["BENCHMARK_PLAIN_TABLE"] = "1"

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    parse_prompt_case,
    run_benchmark,
)

_llm = BENCHMARK_LLM_MODEL
if (_llm is None or not str(_llm).strip()) and "WATERMARK_LLM_ID" in globals():
    _llm = WATERMARK_LLM_ID
if _llm and str(_llm).strip():
    wm.set_llm_model_id(str(_llm).strip())

_cases = (
    [parse_prompt_case(s) for s in BENCHMARK_PROMPT_CASES]
    if BENCHMARK_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)

_console = make_benchmark_console()
_exit = run_benchmark(
    prompt_cases=_cases,
    runs=int(BENCHMARK_RUNS),
    modulus=int(BENCHMARK_MODULUS),
    code_length=int(BENCHMARK_CODE_LENGTH),
    fresh_key_per_trial=not BENCHMARK_REUSE_KEY,
    console=_console,
    llm_model_id=str(_llm).strip() if _llm and str(_llm).strip() else None,
    wm_bit_redundancy=int(BENCHMARK_WM_BIT_REDUNDANCY),
)

print("\nrun_benchmark exit code:", _exit, "(0 = strict protocol passed on every run)")


code_length=500  modulus=1024  runs=5  keys=fresh per run  llm='meta-llama/Llama-3.2-1B-Instruct'

Output()


Per-prompt averages over runs
prompt_id                   runs  master    open  accept  rej_ok    BER%     x==   KPI/x      t_bl      t_wm     t_key     t_det
--------------------------------------------------------------------------------------------------------------------------------
patriots_qbs                   5     4/5     4/5     4/5     3/5   34.40     5/5     2/5    10.322    10.551    0.0089    2.3051

Legend: master/open/accept = successes/runs; rej_ok = correct reject (expect reject=False); BER% = mean bit error vs embedded PRC (master recovery); x== = runs with full x match; KPI/x = full KPI successes among x-matched runs (n/a if no x match); t_bl / t_wm / t_key / t_det = mean seconds.

Exiting with code 1: at least one run did not meet all KPI checks (master + unconstrained + policy-accept must succeed; policy-reject must fail).
  patriots_qbs: master 4/5, open 4/5, accept 4/5, reject_false_positive 2/5

Total wall time (sum over every benchmark run; all prompts)
  bas

SystemExit: 1

## 9. Chart: policy FPR vs PRC random baseline over code length

This section **does not re-run** the single-length cell in section 8; it sweeps **several logical PRC code lengths** and, for each length:

- **Scheme FPR (micro):** pooled false positive rate for per-vocabulary **`detect`** vs gold *label in verify-time NLI-active set* — same definition as the **FPR** column in section 8 (all runs, and a second curve restricted to runs with **encode `attr_x` == verify `derive_x`** when that subset has decisions).
- **PRC random:** Monte Carlo rate that **`prc.detect`** accepts **i.i.d. fair bits** against one fixed random PRC key (same construction as `testing.py`: randomness on the key draw and on each trial’s bit vector).

**Cost:** one full benchmark run **per code length** (same cost order as section 8 × number of lengths). The PRC-only curve is cheap. Tune **`FPR_CHART_CODE_LENGTHS`** and **`FPR_CHART_RUNS`** before running.

The cell writes **`benchmark_fpr_vs_code_length.json`** (numeric series) and **`benchmark_fpr_vs_code_length.png`** under **`PROJECT_ROOT`** for export. Colab already includes **matplotlib**; in a minimal venv, `pip install matplotlib` first.


In [ ]:
import json
import os
import random
import sys
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError("Install matplotlib to render/export charts (e.g. pip install matplotlib).") from e

os.chdir(PROJECT_ROOT)
_root = Path(str(PROJECT_ROOT))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    micro_fpr_wilson,
    parse_prompt_case,
    prc_random_detect_positive_rate,
    run_benchmark_with_summary,
    wilson_score_interval,
)

# --- sweep (edit before running: each length is a full benchmark) ---
FPR_CHART_CODE_LENGTHS = [128, 256, 384]
FPR_CHART_RUNS = 3
FPR_CHART_MODULUS = 1024
FPR_CHART_WM_BIT_REDUNDANCY = 1
FPR_CHART_REUSE_KEY = False
FPR_CHART_PRC_MONTE_CARLO = 100000
FPR_CHART_RNG = random.Random(42)

# Same prompt / LLM conventions as section 8
FPR_CHART_PROMPT_CASES: list[str] = []
FPR_CHART_LLM_MODEL: str | None = None

_llm = FPR_CHART_LLM_MODEL
if (_llm is None or not str(_llm).strip()) and "WATERMARK_LLM_ID" in globals():
    _llm = WATERMARK_LLM_ID
if _llm and str(_llm).strip():
    wm.set_llm_model_id(str(_llm).strip())

_cases = (
    [parse_prompt_case(s) for s in FPR_CHART_PROMPT_CASES]
    if FPR_CHART_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)

_console = make_benchmark_console()

lengths: list[int] = []
scheme_fpr_all: list[float] = []
scheme_fpr_xmatch: list[float] = []
scheme_fpr_all_lo: list[float] = []
scheme_fpr_all_hi: list[float] = []
scheme_fpr_xmatch_lo: list[float] = []
scheme_fpr_xmatch_hi: list[float] = []
prc_random_fpr: list[float] = []
prc_random_fpr_lo: list[float] = []
prc_random_fpr_hi: list[float] = []
exit_codes: list[int] = []

for L in FPR_CHART_CODE_LENGTHS:
    print(f"--- code_length={L}: PRC random baseline ({FPR_CHART_PRC_MONTE_CARLO} trials) ---")
    r_rand, fp_mc = prc_random_detect_positive_rate(
        int(L), int(FPR_CHART_PRC_MONTE_CARLO), rng=FPR_CHART_RNG
    )
    prc_random_fpr.append(float(r_rand))
    pl, ph = wilson_score_interval(fp_mc, int(FPR_CHART_PRC_MONTE_CARLO))
    prc_random_fpr_lo.append(pl)
    prc_random_fpr_hi.append(ph)

    print(f"--- code_length={L}: benchmark ({FPR_CHART_RUNS} runs / prompt) ---")
    ex, summary = run_benchmark_with_summary(
        prompt_cases=_cases,
        runs=int(FPR_CHART_RUNS),
        modulus=int(FPR_CHART_MODULUS),
        code_length=int(L),
        fresh_key_per_trial=not FPR_CHART_REUSE_KEY,
        console=_console,
        llm_model_id=str(_llm).strip() if _llm and str(_llm).strip() else None,
        wm_bit_redundancy=int(FPR_CHART_WM_BIT_REDUNDANCY),
        quiet=True,
    )
    lengths.append(int(L))
    exit_codes.append(int(ex))

    fa, fa_lo, fa_hi = micro_fpr_wilson(summary.roll, summary.prompt_cases)
    fx, fx_lo, fx_hi = micro_fpr_wilson(summary.roll_xmatch, summary.prompt_cases)
    scheme_fpr_all.append(float(fa) if fa >= 0.0 else float("nan"))
    scheme_fpr_xmatch.append(float(fx) if fx >= 0.0 else float("nan"))
    scheme_fpr_all_lo.append(fa_lo if fa >= 0.0 else float("nan"))
    scheme_fpr_all_hi.append(fa_hi if fa >= 0.0 else float("nan"))
    scheme_fpr_xmatch_lo.append(fx_lo if fx >= 0.0 else float("nan"))
    scheme_fpr_xmatch_hi.append(fx_hi if fx >= 0.0 else float("nan"))

    print(
        f"code_length={L}  exit={ex}  FPR_all={scheme_fpr_all[-1]:.6g}  "
        f"FPR_xmatch={scheme_fpr_xmatch[-1]:.6g}  PRC_random={prc_random_fpr[-1]:.6g}"
    )

_json_path = _root / "benchmark_fpr_vs_code_length.json"
_png_path = _root / "benchmark_fpr_vs_code_length.png"
_png_path_ci = _root / "benchmark_fpr_vs_code_length_ci.png"

payload = {
    "code_lengths": lengths,
    "scheme_fpr_all_runs": scheme_fpr_all,
    "scheme_fpr_x_matched_runs_only": scheme_fpr_xmatch,
    "prc_random_detect_rate": prc_random_fpr,
    "scheme_fpr_all_ci_low": scheme_fpr_all_lo,
    "scheme_fpr_all_ci_high": scheme_fpr_all_hi,
    "scheme_fpr_x_matched_ci_low": scheme_fpr_xmatch_lo,
    "scheme_fpr_x_matched_ci_high": scheme_fpr_xmatch_hi,
    "prc_random_detect_rate_ci_low": prc_random_fpr_lo,
    "prc_random_detect_rate_ci_high": prc_random_fpr_hi,
    "ci_note": "Wilson score ~95% intervals; scheme from pooled micro confusion counts; PRC Monte Carlo",
    "benchmark_exit_codes": exit_codes,
    "runs_per_prompt": int(FPR_CHART_RUNS),
    "prc_monte_carlo_trials": int(FPR_CHART_PRC_MONTE_CARLO),
    "modulus": int(FPR_CHART_MODULUS),
    "wm_bit_redundancy": int(FPR_CHART_WM_BIT_REDUNDANCY),
}
_json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote", _json_path)

def _asym_errbar(y_vals, lo, hi):
    el, eh = [], []
    for y, a, b in zip(y_vals, lo, hi):
        if y != y or a != a or b != b:
            el.append(float("nan"))
            eh.append(float("nan"))
        else:
            el.append(y - a)
            eh.append(b - y)
    return [el, eh]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lengths, scheme_fpr_all, "o-", label="Scheme FPR (all runs)")
ax.plot(lengths, scheme_fpr_xmatch, "s--", label="Scheme FPR (x matched runs only)")
ax.plot(lengths, prc_random_fpr, "^-", label="PRC Baseline")
ax.set_xlabel("Output Length (tokens)")
ax.set_ylabel("False Positive Rate")
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(_png_path, dpi=150)
plt.show()
print("Wrote", _png_path)

fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.errorbar(
    lengths,
    scheme_fpr_all,
    yerr=_asym_errbar(scheme_fpr_all, scheme_fpr_all_lo, scheme_fpr_all_hi),
    fmt="o-",
    capsize=4,
    label="Scheme FPR (all runs)",
)
ax2.errorbar(
    lengths,
    scheme_fpr_xmatch,
    yerr=_asym_errbar(scheme_fpr_xmatch, scheme_fpr_xmatch_lo, scheme_fpr_xmatch_hi),
    fmt="s--",
    capsize=4,
    label="Scheme FPR (x matched runs only)",
)
ax2.errorbar(
    lengths,
    prc_random_fpr,
    yerr=_asym_errbar(prc_random_fpr, prc_random_fpr_lo, prc_random_fpr_hi),
    fmt="^-",
    capsize=4,
    label="PRC Baseline",
)
ax2.set_xlabel("Output Length (tokens)")
ax2.set_ylabel("False Positive Rate")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="best")
fig2.tight_layout()
fig2.savefig(_png_path_ci, dpi=150)
plt.show()
print("Wrote", _png_path_ci)


## 10. Chart: policy TPR vs WM bit redundancy

Sweeps **`WM_BIT_REDUNDANCY`** (logical PRC bit repeats on the token channel; same as **`app.py`** / section 8) at the fixed **logical PRC code length 100**. For each redundancy value **1, 3, 5**, runs the same benchmark as section 8 (via **`run_benchmark_with_summary(..., quiet=True)`**) and plots **micro-averaged TPR** for policy **`detect`** vs gold *label in verify-time NLI-active set* — **all runs** and **x_matched runs only** (encode **`attr_x`** == verify **`derive_x`**).

**Cost:** one full benchmark per redundancy value. Tune **`TPR_CHART_RUNS`** before running.

Exports **`benchmark_tpr_vs_wm_bit_redundancy.json`** and **`benchmark_tpr_vs_wm_bit_redundancy.png`** under **`PROJECT_ROOT`**.


In [ ]:
import json
import os
import sys
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError("Install matplotlib to render/export charts (e.g. pip install matplotlib).") from e

os.chdir(PROJECT_ROOT)
_root = Path(str(PROJECT_ROOT))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    micro_tpr_wilson,
    parse_prompt_case,
    run_benchmark_with_summary,
)

# --- sweep: redundancy at fixed code length (edit before running) ---
TPR_CHART_WM_BIT_REDUNDANCY_VALUES = [1, 3, 5]
TPR_CHART_CODE_LENGTH = 100
TPR_CHART_RUNS = 3
TPR_CHART_MODULUS = 1024
TPR_CHART_REUSE_KEY = False

TPR_CHART_PROMPT_CASES: list[str] = []
TPR_CHART_LLM_MODEL: str | None = None

_llm = TPR_CHART_LLM_MODEL
if (_llm is None or not str(_llm).strip()) and "WATERMARK_LLM_ID" in globals():
    _llm = WATERMARK_LLM_ID
if _llm and str(_llm).strip():
    wm.set_llm_model_id(str(_llm).strip())

_cases = (
    [parse_prompt_case(s) for s in TPR_CHART_PROMPT_CASES]
    if TPR_CHART_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)

_console = make_benchmark_console()

redundancies: list[int] = []
tpr_all: list[float] = []
tpr_xmatch: list[float] = []
tpr_all_lo: list[float] = []
tpr_all_hi: list[float] = []
tpr_xmatch_lo: list[float] = []
tpr_xmatch_hi: list[float] = []
exit_codes: list[int] = []

for R in TPR_CHART_WM_BIT_REDUNDANCY_VALUES:
    print(f"--- wm_bit_redundancy={R}  code_length={TPR_CHART_CODE_LENGTH} ---")
    ex, summary = run_benchmark_with_summary(
        prompt_cases=_cases,
        runs=int(TPR_CHART_RUNS),
        modulus=int(TPR_CHART_MODULUS),
        code_length=int(TPR_CHART_CODE_LENGTH),
        fresh_key_per_trial=not TPR_CHART_REUSE_KEY,
        console=_console,
        llm_model_id=str(_llm).strip() if _llm and str(_llm).strip() else None,
        wm_bit_redundancy=int(R),
        quiet=True,
    )
    redundancies.append(int(R))
    exit_codes.append(int(ex))

    ta, ta_lo, ta_hi = micro_tpr_wilson(summary.roll, summary.prompt_cases)
    tx, tx_lo, tx_hi = micro_tpr_wilson(summary.roll_xmatch, summary.prompt_cases)
    tpr_all.append(float(ta) if ta >= 0.0 else float("nan"))
    tpr_xmatch.append(float(tx) if tx >= 0.0 else float("nan"))
    tpr_all_lo.append(ta_lo if ta >= 0.0 else float("nan"))
    tpr_all_hi.append(ta_hi if ta >= 0.0 else float("nan"))
    tpr_xmatch_lo.append(tx_lo if tx >= 0.0 else float("nan"))
    tpr_xmatch_hi.append(tx_hi if tx >= 0.0 else float("nan"))

    print(
        f"wm_bit_redundancy={R}  exit={ex}  TPR_all={tpr_all[-1]:.6g}  TPR_xmatch={tpr_xmatch[-1]:.6g}"
    )

_json_path = _root / "benchmark_tpr_vs_wm_bit_redundancy.json"
_png_path = _root / "benchmark_tpr_vs_wm_bit_redundancy.png"
_png_path_ci = _root / "benchmark_tpr_vs_wm_bit_redundancy_ci.png"

payload = {
    "wm_bit_redundancy": redundancies,
    "tpr_all_runs": tpr_all,
    "tpr_x_matched_runs_only": tpr_xmatch,
    "tpr_all_runs_ci_low": tpr_all_lo,
    "tpr_all_runs_ci_high": tpr_all_hi,
    "tpr_x_matched_ci_low": tpr_xmatch_lo,
    "tpr_x_matched_ci_high": tpr_xmatch_hi,
    "ci_note": "Wilson score ~95% from pooled micro confusion counts",
    "benchmark_exit_codes": exit_codes,
    "code_length": int(TPR_CHART_CODE_LENGTH),
    "runs_per_prompt": int(TPR_CHART_RUNS),
    "modulus": int(TPR_CHART_MODULUS),
}
_json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote", _json_path)

def _asym_errbar(y_vals, lo, hi):
    el, eh = [], []
    for y, a, b in zip(y_vals, lo, hi):
        if y != y or a != a or b != b:
            el.append(float("nan"))
            eh.append(float("nan"))
        else:
            el.append(y - a)
            eh.append(b - y)
    return [el, eh]

output_lens = [int(TPR_CHART_CODE_LENGTH) * int(R) for R in redundancies]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(redundancies, tpr_all, "o-", label="All runs")
ax.plot(redundancies, tpr_xmatch, "s--", label="X matched runs only")
ax.axhline(1.0, color="gray", linestyle=":", linewidth=1.0)
ax.set_xticks(redundancies)
ax.set_xticklabels([f"{R}\n({L} tokens)" for R, L in zip(redundancies, output_lens)])
ax.set_xlabel("Redundancy/Output Length")
ax.set_ylabel("True Positive Rate")
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(_png_path, dpi=150)
plt.show()
print("Wrote", _png_path)

fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.errorbar(
    redundancies,
    tpr_all,
    yerr=_asym_errbar(tpr_all, tpr_all_lo, tpr_all_hi),
    fmt="o-",
    capsize=4,
    label="All runs",
)
ax2.errorbar(
    redundancies,
    tpr_xmatch,
    yerr=_asym_errbar(tpr_xmatch, tpr_xmatch_lo, tpr_xmatch_hi),
    fmt="s--",
    capsize=4,
    label="X matched runs only",
)
ax2.axhline(1.0, color="gray", linestyle=":", linewidth=1.0)
ax2.set_xticks(redundancies)
ax2.set_xticklabels([f"{R}\n({L} tokens)" for R, L in zip(redundancies, output_lens)])
ax2.set_xlabel("Redundancy/Output Length")
ax2.set_ylabel("True Positive Rate")
ax2.set_ylim(-0.02, 1.02)
ax2.grid(True, alpha=0.3)
ax2.legend(loc="best")
fig2.tight_layout()
fig2.savefig(_png_path_ci, dpi=150)
plt.show()
print("Wrote", _png_path_ci)


## 11. Matrix: constrained-key detection rate conditioned on attributed labels

Builds a **`|V| × |V|`** matrix over the closed vocabulary (**rows = constrained key label used for `detect`**, **columns = labels attributed by verify-time `derive_x`**).

Per benchmark trial (with multi-label attribution allowed), the update is done in **two explicit passes**:
1. Let **`A`** be the set of attributed labels for that trial.
2. **Denominator pass:** for every `c in A`, increment the denominator of **every row** in that column: `den[r,c] += 1` for all row labels `r`.
3. **Numerator pass:** run constrained detection for every row label `r`; if detection for row `r` is positive, increment only that row across the same attributed columns: `num[r,c] += 1` for all `c in A`.

So each cell is a conditional positive-detection rate:
- rate `[r,c] = num[r,c] / den[r,c]` (or n/a when `den[r,c] = 0`)

The output includes both:
- **percentage matrix**
- **raw fraction matrix** (`n/d`)

The cell exports CSV/JSON artifacts under **`PROJECT_ROOT`** for easy Colab export/download.

In [ ]:
import json
import os
import sys
from pathlib import Path

try:
    import pandas as pd
except ImportError as e:
    raise ImportError("Install pandas for matrix display/export (e.g. pip install pandas).") from e

os.chdir(PROJECT_ROOT)
_root = Path(str(PROJECT_ROOT))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    parse_prompt_case,
    run_benchmark_label_conditioned_matrix,
)

# --- matrix benchmark parameters (same style as section 8) ---
MATRIX_RUNS = 3
MATRIX_CODE_LENGTH = 300
MATRIX_WM_BIT_REDUNDANCY = 1
MATRIX_MODULUS = 1024
MATRIX_REUSE_KEY = False

MATRIX_PROMPT_CASES: list[str] = []
MATRIX_LLM_MODEL: str | None = None

_llm = MATRIX_LLM_MODEL
if (_llm is None or not str(_llm).strip()) and "WATERMARK_LLM_ID" in globals():
    _llm = WATERMARK_LLM_ID
if _llm and str(_llm).strip():
    wm.set_llm_model_id(str(_llm).strip())

_cases = (
    [parse_prompt_case(s) for s in MATRIX_PROMPT_CASES]
    if MATRIX_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)

_console = make_benchmark_console()
_exit, _mx = run_benchmark_label_conditioned_matrix(
    prompt_cases=_cases,
    runs=int(MATRIX_RUNS),
    modulus=int(MATRIX_MODULUS),
    code_length=int(MATRIX_CODE_LENGTH),
    fresh_key_per_trial=not MATRIX_REUSE_KEY,
    console=_console,
    llm_model_id=str(_llm).strip() if _llm and str(_llm).strip() else None,
    wm_bit_redundancy=int(MATRIX_WM_BIT_REDUNDANCY),
    quiet=True,
)

_vocab = list(_mx.vocab)

num_df = pd.DataFrame(_mx.numerators).T.loc[_vocab, _vocab]
den_df = pd.DataFrame(_mx.denominators).T.loc[_vocab, _vocab]

pct_df = (num_df / den_df.replace(0, pd.NA)) * 100.0
frac_df = num_df.astype(str) + "/" + den_df.astype(str)

print(
    f"matrix exit code: {_exit} (0=strict protocol checks passed on every run); "
    f"runs/prompt={MATRIX_RUNS}; |V|={len(_vocab)}"
)
print("\nPercentage matrix (rows=constrained key label, cols=attributed label):")
display(pct_df.round(2))

print("\nRaw fraction matrix n/d (same row/col semantics):")
display(frac_df)

prefix = _root / "benchmark_label_conditioned_detection_matrix"

pct_df.to_csv(prefix.with_name(prefix.name + "_percent.csv"), encoding="utf-8")
frac_df.to_csv(prefix.with_name(prefix.name + "_fraction.csv"), encoding="utf-8")
num_df.to_csv(prefix.with_name(prefix.name + "_numerator.csv"), encoding="utf-8")
den_df.to_csv(prefix.with_name(prefix.name + "_denominator.csv"), encoding="utf-8")

json_payload = {
    "vocab": _vocab,
    "rows_constrained_key_label": _vocab,
    "cols_attributed_label": _vocab,
    "numerators": _mx.numerators,
    "denominators": _mx.denominators,
    "rates": _mx.rates,
    "matrix_exit_code": int(_exit),
    "strict_protocol_ok": bool(_mx.strict_protocol_ok),
    "runs_per_prompt": int(_mx.runs_per_prompt),
    "code_length": int(_mx.code_length),
    "wm_bit_redundancy": int(_mx.wm_bit_redundancy),
    "modulus": int(_mx.modulus),
    "prompt_case_ids": [sid for sid, _ in _mx.prompt_cases],
}
prefix.with_name(prefix.name + ".json").write_text(
    json.dumps(json_payload, indent=2), encoding="utf-8"
)

print("\nWrote:")
print(" -", prefix.with_name(prefix.name + "_percent.csv"))
print(" -", prefix.with_name(prefix.name + "_fraction.csv"))
print(" -", prefix.with_name(prefix.name + "_numerator.csv"))
print(" -", prefix.with_name(prefix.name + "_denominator.csv"))
print(" -", prefix.with_name(prefix.name + ".json"))

## 12. Matrix: constrained-key detection rate by benchmark prompt

Same pipeline and cell semantics as **section 11**, except the matrix is **`|V| × |P|`**:

- **Rows:** constrained key label used for `detect` (closed vocabulary).
- **Columns:** one column per **benchmark prompt id** (the `id` in each `id:prompt` case), in the order given in **`PROMPT_MATRIX_PROMPT_CASES`** or the default list.

Per trial on prompt id `p`, only if verify-time attribution is non-empty (same skip as section 11):
1. **Denominator:** `den[r, p] += 1` for every vocabulary row `r`.
2. **Numerator:** if `detect` for row `r` is positive, `num[r, p] += 1`.

Exports **`benchmark_prompt_conditioned_detection_matrix_*`** under **`PROJECT_ROOT`**.

In [ ]:
import json
import os
import sys
from pathlib import Path

try:
    import pandas as pd
except ImportError as e:
    raise ImportError("Install pandas for matrix display/export (e.g. pip install pandas).") from e

os.chdir(PROJECT_ROOT)
_root = Path(str(PROJECT_ROOT))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    parse_prompt_case,
    run_benchmark_prompt_conditioned_matrix,
)

PROMPT_MATRIX_RUNS = 3
PROMPT_MATRIX_CODE_LENGTH = 300
PROMPT_MATRIX_WM_BIT_REDUNDANCY = 1
PROMPT_MATRIX_MODULUS = 1024
PROMPT_MATRIX_REUSE_KEY = False

PROMPT_MATRIX_PROMPT_CASES: list[str] = []
PROMPT_MATRIX_LLM_MODEL: str | None = None

_llm = PROMPT_MATRIX_LLM_MODEL
if (_llm is None or not str(_llm).strip()) and "WATERMARK_LLM_ID" in globals():
    _llm = WATERMARK_LLM_ID
if _llm and str(_llm).strip():
    wm.set_llm_model_id(str(_llm).strip())

_cases = (
    [parse_prompt_case(s) for s in PROMPT_MATRIX_PROMPT_CASES]
    if PROMPT_MATRIX_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)

_console = make_benchmark_console()
_exit_pm, _pmx = run_benchmark_prompt_conditioned_matrix(
    prompt_cases=_cases,
    runs=int(PROMPT_MATRIX_RUNS),
    modulus=int(PROMPT_MATRIX_MODULUS),
    code_length=int(PROMPT_MATRIX_CODE_LENGTH),
    fresh_key_per_trial=not PROMPT_MATRIX_REUSE_KEY,
    console=_console,
    llm_model_id=str(_llm).strip() if _llm and str(_llm).strip() else None,
    wm_bit_redundancy=int(PROMPT_MATRIX_WM_BIT_REDUNDANCY),
    quiet=True,
)

_vocab = list(_pmx.vocab)
_cols = list(_pmx.column_prompt_ids)

num_df = pd.DataFrame(_pmx.numerators).T.loc[_vocab, _cols]
den_df = pd.DataFrame(_pmx.denominators).T.loc[_vocab, _cols]

pct_df = (num_df / den_df.replace(0, pd.NA)) * 100.0
frac_df = num_df.astype(str) + "/" + den_df.astype(str)

print(
    f"prompt-matrix exit code: {_exit_pm} (0=strict protocol checks passed on every run); "
    f"runs/prompt={PROMPT_MATRIX_RUNS}; |V|={len(_vocab)}; |P|={len(_cols)}"
)
print("\nPercentage matrix (rows=constrained key label, cols=prompt id):")
display(pct_df.round(2))

print("\nRaw fraction matrix n/d:")
display(frac_df)

prefix = _root / "benchmark_prompt_conditioned_detection_matrix"

pct_df.to_csv(prefix.with_name(prefix.name + "_percent.csv"), encoding="utf-8")
frac_df.to_csv(prefix.with_name(prefix.name + "_fraction.csv"), encoding="utf-8")
num_df.to_csv(prefix.with_name(prefix.name + "_numerator.csv"), encoding="utf-8")
den_df.to_csv(prefix.with_name(prefix.name + "_denominator.csv"), encoding="utf-8")

json_payload = {
    "vocab": _vocab,
    "rows_constrained_key_label": _vocab,
    "cols_prompt_id": _cols,
    "numerators": _pmx.numerators,
    "denominators": _pmx.denominators,
    "rates": _pmx.rates,
    "matrix_exit_code": int(_exit_pm),
    "strict_protocol_ok": bool(_pmx.strict_protocol_ok),
    "runs_per_prompt": int(_pmx.runs_per_prompt),
    "code_length": int(_pmx.code_length),
    "wm_bit_redundancy": int(_pmx.wm_bit_redundancy),
    "modulus": int(_pmx.modulus),
    "prompt_case_ids": [sid for sid, _ in _pmx.prompt_cases],
}
prefix.with_name(prefix.name + ".json").write_text(
    json.dumps(json_payload, indent=2), encoding="utf-8"
)

print("\nWrote:")
print(" -", prefix.with_name(prefix.name + "_percent.csv"))
print(" -", prefix.with_name(prefix.name + "_fraction.csv"))
print(" -", prefix.with_name(prefix.name + "_numerator.csv"))
print(" -", prefix.with_name(prefix.name + "_denominator.csv"))
print(" -", prefix.with_name(prefix.name + ".json"))

## 13. Run `app.py` with a custom prompt

Use this section for quick single-run experiments with arbitrary prompts from the notebook (without editing `app.py`).

It imports `app` as a module, overrides `app.PROMPT` (and optional run params), and calls `app.main()` directly so you keep the exact app protocol/logging behavior.

In [ ]:
import importlib
import os
import sys
from pathlib import Path

os.chdir(PROJECT_ROOT)
_root = Path(str(PROJECT_ROOT))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import watermarking as wm
import app as app_mod

# --- edit these for ad-hoc single-run testing ---
APP_CUSTOM_PROMPT = "Explain how advances in battery chemistry changed electric vehicle adoption."
APP_CUSTOM_CODE_LENGTH = 100
APP_CUSTOM_WM_BIT_REDUNDANCY = 3
APP_CUSTOM_MODULUS = 1024
APP_CUSTOM_LLM_MODEL: str | None = None

_llm = APP_CUSTOM_LLM_MODEL
if (_llm is None or not str(_llm).strip()) and "WATERMARK_LLM_ID" in globals():
    _llm = WATERMARK_LLM_ID
if _llm and str(_llm).strip():
    wm.set_llm_model_id(str(_llm).strip())

# Reload module to reset any prior monkeypatches, then patch run-time constants.
app_mod = importlib.reload(app_mod)
app_mod.PROMPT = str(APP_CUSTOM_PROMPT)
app_mod.CODE_LENGTH = int(APP_CUSTOM_CODE_LENGTH)
app_mod.WM_BIT_REDUNDANCY = int(APP_CUSTOM_WM_BIT_REDUNDANCY)
app_mod.MODULUS = int(APP_CUSTOM_MODULUS)
app_mod.LLM_MODEL_ID = str(_llm).strip() if _llm and str(_llm).strip() else None

print("Running app.main() with custom prompt...")
print("  prompt:", app_mod.PROMPT)
print(
    "  params:",
    {
        "code_length": app_mod.CODE_LENGTH,
        "wm_bit_redundancy": app_mod.WM_BIT_REDUNDANCY,
        "modulus": app_mod.MODULUS,
        "llm": app_mod.LLM_MODEL_ID or wm.MODEL_ID,
    },
)

_app_exit = app_mod.main()
print("\napp.main() exit code:", _app_exit, "(0 = strict protocol passed on every check)")